code for generating simulated data and processing them into the bronze layer tables

Initialization

In [0]:
catalog_name = "workspace"
database_name = "amazon_fullfilment_bronze"



In [0]:
bronze_volume_path ="/Volumes/workspace/amazon_fullfilment_bronze/landing_zone/"
source_volume_path = "/Volumes/workspace/default/amazon_fullfilment/source/"

In [0]:
%run ./Includes/Bronze_generate_data


Checking for source data

In [0]:
# Checking if source files (Orders) exist before proceeding

try: 
  orders_files_df_check = dbutils.fs.ls(source_volume_path + "orders")
  if not orders_files_df_check:
    raise Exception("No Orders file(s) found.  Aborting ...")
  print(f"Success: Found Orders files. Proceeding to processing.")
except Exception as e:
  if "java.io.FileNotFoundException" in str(e):
    raise Exception(f"CRITICAL: Directory for Orders data does not exist: {source_volume_path}/orders") from e
  else:
    raise e


Table inserts

In [0]:
# Insert into Bronze layer Robots table
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Define exactly what the data should look like
robot_schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("RobotID", StringType(), True),
    StructField("BatteryLevel", DoubleType(), True),
    StructField("Status", StringType(), True),
    StructField("LocationZone", StringType(), True),
    StructField("Processed_date", DateType(), True),
    StructField("LastUpdated", TimestampType(), True)
])


# 1. Define your paths

robots_checkpoint_path = f"{bronze_volume_path}robots"

# 2. Set up the Auto Loader Stream
bronze_robot_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv") # Assuming you save mock data as Delta
  .option("header", "true")
  .option("cloudFiles.schemaLocation", robots_checkpoint_path) 
  .option("dateFormat", "yyyy-MM-dd")
  .schema(robot_schema)
  .load(f"{source_volume_path}/robots"))

# 3. Write it to a permanent Bronze Table
(bronze_robot_stream.writeStream
  .option("checkpointLocation", robots_checkpoint_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.robots"))

In [0]:
# Insert into Bronze layer customer and address tables
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType


# Define exactly what the data should look like
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True)
])

address_schema = StructType([
    StructField("address_id", StringType(),True),
    StructField("customer_id", StringType(), True),
    StructField("street", StringType(), True),
    StructField("city", StringType(), True),
    StructField("province", StringType(), True),
    StructField("postal_code", StringType(), True)
])


# 1. Define your paths

bronze_customer_path = f"{bronze_volume_path}customers"
bronze_address_path = f"{bronze_volume_path}addresses"

# 2. Set up the Auto Loader Stream for customers
bronze_customers_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true") 
  .option("cloudFiles.schemaLocation", bronze_customer_path) 
  .schema(customer_schema)
  .load(f"{source_volume_path}customers"))

# 3. Write it to a permanent Bronze Table for customers
(bronze_customers_stream.writeStream
  .option("checkpointLocation", bronze_customer_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.customers"))


# 4. Set up the Auto Loader Stream for addreses
bronze_addresses_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")  
  .option("cloudFiles.schemaLocation", bronze_address_path) 
  .schema(address_schema)
  .load(f"{source_volume_path}addresses"))

# 5. Write it to a permanent Bronze Table for addresses
(bronze_addresses_stream.writeStream
  .option("checkpointLocation", bronze_address_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.addresses"))


In [0]:
# Insert into Bronze layer Products table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Define exactly what the data should look like
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("sku", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("weight_kg", DoubleType(), True)
])


# 1. Define your paths
bronze_products_path = f"{bronze_volume_path}products"

# 2. Set up the Auto Loader Stream
bronze_products_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv") # Assuming you save mock data as Delta
  .option("header", "true") 
  .option("cloudFiles.schemaLocation", bronze_products_path) 
  .schema(products_schema)
  .load(f"{source_volume_path}products"))

# 3. Write it to a permanent Bronze Table
(bronze_products_stream.writeStream
  .option("checkpointLocation", bronze_products_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.products"))

In [0]:
%sql
SELECT current_catalog(), current_schema();

In [0]:
%sql
select * from csv.`/Volumes/workspace/default/amazon_fullfilment/source/inventory`

In [0]:
# Insert into Bronze layer shelves and inventory tables
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType


# Define exactly what the data should look like
shelves_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("curent_robot_id", StringType(), True),
    StructField("max_weight_capacity", DoubleType(), True),
    StructField("status", StringType(), True)
])

inventory_schema = StructType([
    StructField("shelf_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("associated_order_id", StringType(), True)
])


# 1. Define your paths
bronze_shelves_path = f"{bronze_volume_path}shelves"
bronze_inventory_path = f"{bronze_volume_path}inventory"


# 2. Set up the Auto Loader Stream for shelves
bronze_shelves_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")  
  .option("cloudFiles.schemaLocation", bronze_shelves_path) 
  .schema(shelves_schema)
  .load(f"{source_volume_path}shelves"))

# 3. Write it to a permanent Bronze Table for shelves
(bronze_shelves_stream.writeStream
  .option("checkpointLocation", bronze_shelves_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.shelves"))


# 4. Set up the Auto Loader Stream for addreses
bronze_inventory_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")  
  .option("cloudFiles.schemaLocation", bronze_inventory_path) 
  .schema(inventory_schema)
  .load(f"{source_volume_path}inventory"))

# 5. Write it to a permanent Bronze Table for addresses
(bronze_inventory_stream.writeStream
  .option("checkpointLocation", bronze_inventory_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.inventory"))

In [0]:
# Insert into Bronze layer orders table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Define exactly what the data should look like
order_schema = StructType([
    StructField("orderid", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("orderdate", DateType(), True),
    StructField("status", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])


# 1. Define your paths
bronze_orders_path = f"{bronze_volume_path}orders"


# 2. Set up the Auto Loader Stream
bronze_orders_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.schemaLocation", bronze_orders_path) # Auto-infer schema here
  .option("dateFormat", "yyyy-MM-dd")
  .schema(order_schema)
  .load(f"{source_volume_path}orders"))

# 3. Write it to a permanent Bronze Table
(bronze_orders_stream.writeStream
  .option("checkpointLocation", bronze_orders_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.orders"))

 

In [0]:
# Insert into Bronze layer order_items table
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, IntegerType

# Define exactly what the data should look like
order_item_schema = StructType([
    StructField("orderitemid", StringType(), True),
    StructField("orderid", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("Quantity", IntegerType(), True)
])


# 1. Define your paths
bronze_order_items_path = f"{bronze_volume_path}order_items"

# 2. Set up the Auto Loader Stream
bronze_order_items_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true") 
  .option("cloudFiles.schemaLocation", bronze_order_items_path) 
  .schema(order_item_schema)
  .load(f"{source_volume_path}order_items"))

# 3. Write it to a permanent Bronze Table
(bronze_order_items_stream.writeStream
  .option("checkpointLocation", bronze_order_items_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.order_items"))

In [0]:
# Insert into Bronze layer Bin table
# NOT NEEDED ANYMORE
 
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType, LongType

# Define exactly what the data should look like
bin_schema = StructType([
    StructField("bin_id", StringType(), True),
    StructField("current_order_id", StringType(), True),
    StructField("item_count", LongType(), True),
    StructField("bin_status", StringType(), True),
    StructField("current_weight", DoubleType(), True),
    StructField("last_employee_id", StringType(), True),
    StructField("updated_timestamp", TimestampType(), True)
])

# 1. Define your paths
bronze_bin_path = f"{bronze_volume_path}bin"

# 2. Set up the Auto Loader Stream
bronze_bin_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.schemaLocation", bronze_bin_path) 
  .schema(bin_schema)
  .load(f"{source_volume_path}bin"))

# 3. Write it to a permanent Bronze Table
(bronze_bin_stream.writeStream
  .option("checkpointLocation", bronze_bin_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.bin"))

In [0]:
# Insert into Bronze layer shipment table

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, DateType

# Define exactly what the data should look like
shipment_schema = StructType([
    StructField("shipmentid", StringType(), True),
    StructField("orderid", StringType(), True),
    StructField("trackingnumber", StringType(), True),
    StructField("shippinglabelurl", StringType(), True),
    StructField("vehicleid", StringType(), True)
])

# 1. Define your paths
bronze_shipment_path = f"{bronze_volume_path}shipment"

# 2. Set up the Auto Loader Stream
bronze_shipment_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("cloudFiles.schemaLocation", bronze_shipment_path) 
  .schema(shipment_schema)
  .load(f"{source_volume_path}shipment"))

# 3. Write it to a permanent Bronze Table
(bronze_shipment_stream.writeStream
  .option("checkpointLocation", bronze_shipment_path)
  .outputMode("append")
  .trigger(availableNow=True)
  .toTable(f"{catalog_name}.{database_name}.shipment"))